# Création de tables KB_PROPOSAL et KB_VERSIONS

In [ ]:
import json
import importlib
import os
import sys

# Rend le package src (à la racine du projet) importable depuis notebooks/
_project_root = os.path.abspath("..")
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.llm import make_llm
from src.graph import build_graph
from src.trace import show_trace
from src.workflows.kb_repair import create_kb_proposal
from src.workflows.kb_repair import generate_kb_suggestion
from src.workflows.kb_repair import approve_proposal
from src.workflows.kb_repair import reject_proposal
from src.workflows.kb_repair import publish_proposal
from src.workflows.kb_repair import rollback_proposal
from snowflake.snowpark.context import get_active_session

In [ ]:
%pip install langgraph langchain-openai rank_bm25

In [ ]:
%%sql -r dataframe_1
CREATE TABLE IF NOT EXISTS PROJECT_DB.PUBLIC.KB_PROPOSALS (
    PROPOSAL_ID VARCHAR,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),

    TICKET_ID VARCHAR,
    QUESTION TEXT,

    PROBLEM_TYPE VARCHAR,
    PROBLEM_DESCRIPTION TEXT,

    SOURCES VARIANT,
    SUGGESTED_CONTENT TEXT,

    STATUS VARCHAR DEFAULT 'pending',

    REVIEWED_AT TIMESTAMP_NTZ,
    REVIEWED_BY VARCHAR,
    REVIEW_COMMENT TEXT,

    PRIMARY KEY (PROPOSAL_ID)
);

In [ ]:
%%sql -r dataframe_2
-- TABLE HISTORIQUE 
CREATE TABLE IF NOT EXISTS PROJECT_DB.PUBLIC.KB_VERSIONS (
    VERSION_ID VARCHAR,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),

    PROPOSAL_ID VARCHAR,

    SOURCE_ID VARCHAR,
    SOURCE_TYPE VARCHAR,

    OLD_CONTENT TEXT,
    NEW_CONTENT TEXT,

    ACTION VARCHAR,

    PRIMARY KEY (VERSION_ID)
);

## Test creation de proposition et génération de suggestion

In [ ]:
session = get_active_session()
llm = make_llm()

graph = build_graph(
    llm=llm,
    session=session
)
with open("../tests/scenarios.json", "r") as f:
    scenarios = json.load(f)["scenarios"]

scenario = scenarios[3] #scenario pour tester  etape 10

initial_state = {
    "ticket": scenario["ticket"],
    "fixtures": scenario.get("fixtures", {}),
    "trace": []
}

result = graph.invoke(initial_state)

print("Décision :", result["decision"])
print("Règle    :", result["decision_rule"])
print("Réponse  :", result["answer"])

show_trace(result["trace"])


In [ ]:

proposal = create_kb_proposal(
    session=session,
    llm=llm,
    state=result
)

print(proposal)

In [ ]:
session.sql("""
    SELECT
        PROPOSAL_ID,
        TICKET_ID,
        PROBLEM_TYPE,
        PROBLEM_DESCRIPTION,
        SUGGESTED_CONTENT,
        SOURCES,
        STATUS,
        CREATED_AT
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    ORDER BY CREATED_AT DESC
""").show()

In [ ]:
proposal_id = proposal["proposal_id"]

session.sql("""
    SELECT
        PROPOSAL_ID,
        QUESTION,
        PROBLEM_DESCRIPTION,
        SUGGESTED_CONTENT,
        STATUS
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[proposal_id]).show()

## Test approuver/rejeter une proposition

In [ ]:

approve_proposal(
    session=session,
    proposal_id=proposal_id,
    reviewer="javier",
    comment="Proposition validée pour test."
)

In [ ]:
session.sql("""
    SELECT
        PROPOSAL_ID,
        STATUS,
        REVIEWED_BY,
        REVIEWED_AT,
        REVIEW_COMMENT
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[proposal_id]).show()

In [ ]:
proposal2 = create_kb_proposal(
    session=session,
    llm=llm,
    state=result
)

other_proposal_id = proposal2["proposal_id"]

In [ ]:

reject_proposal(
    session=session,
    proposal_id=other_proposal_id,
    reviewer="javier",
    comment="Contenu trop vague."
)

In [ ]:
session.sql("""
    SELECT
        PROPOSAL_ID,
        STATUS,
        REVIEWED_BY,
        REVIEWED_AT,
        REVIEW_COMMENT
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[other_proposal_id]).show()

In [ ]:
session.sql("""
    ALTER TABLE PROJECT_DB.PUBLIC.KB_PROPOSALS
    ADD COLUMN IF NOT EXISTS PUBLISHED_SOURCE_ID VARCHAR
""").collect()

session.sql("""
    ALTER TABLE PROJECT_DB.PUBLIC.KB_PROPOSALS
    ADD COLUMN IF NOT EXISTS PUBLISHED_AT TIMESTAMP_NTZ
""").collect()

## Test Publication

In [ ]:

publication = publish_proposal(
    session=session,
    proposal_id=proposal_id
)

print(publication)

In [ ]:
session.sql("""
    SELECT
        PROPOSAL_ID,
        STATUS,
        PUBLISHED_SOURCE_ID,
        PUBLISHED_AT
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[proposal_id]).show()

In [ ]:
source_id = publication["source_id"]

session.sql("""
    SELECT
        FILE_NAME,
        SOURCE_TYPE,
        CONTENT,
        EMBEDDINGS IS NOT NULL AS HAS_EMBEDDING
    FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
    WHERE FILE_NAME = ?
""", params=[source_id]).show()

In [ ]:
session.sql("""
    SELECT
        VERSION_ID,
        PROPOSAL_ID,
        SOURCE_ID,
        ACTION,
        OLD_CONTENT,
        NEW_CONTENT
    FROM PROJECT_DB.PUBLIC.KB_VERSIONS
    WHERE PROPOSAL_ID = ?
""", params=[proposal_id]).show()

## Test Rollback

In [ ]:
rollback = rollback_proposal(
    session=session,
    proposal_id=proposal_id
)

print(rollback)

In [ ]:
session.sql("""
    SELECT STATUS
    FROM PROJECT_DB.PUBLIC.KB_PROPOSALS
    WHERE PROPOSAL_ID = ?
""", params=[proposal_id]).show()

In [ ]:
session.sql("""
    SELECT COUNT(*) AS NB
    FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
    WHERE FILE_NAME = ?
""", params=[source_id]).show()